In [ ]:
!pip install groq python-dotenv numpy tqdm datasets

In [ ]:
from groq import Groq
from dotenv import load_dotenv
from datasets import load_dataset

import os
import time
from tqdm import tqdm
import re
import random
import pprint

from typing import List, Dict, Any

load_dotenv()
random.seed(0)

client = Groq()
gsm8k_dataset = load_dataset("gsm8k", "main")

gsm8k_train = gsm8k_dataset["train"]
gsm8k_test  = gsm8k_dataset["test"]

README.md: 0.00B [00:00, ?B/s]

c:\Users\nk233\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nk233\.cache\huggingface\hub\datasets--gsm8k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [40]:
def generate_response_using_Llama(
        prompt: str,
        model: str = "llama-3.1-8b-instant"
    ):
    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "system",
                    "content": "You are a helpful assistant that solves math problems."
                },
                {
                    "role": "user", 
                    "content": prompt
                }
            ],
            model=model,
            temperature=0.3, ### 수정해도 됩니다!
            stream=False
        )
        return chat_completion.choices[0].message.content
    
    except Exception as e:
        print(f"API call error: {str(e)}")
        return None

#### 응답 잘 나오는지 확인해보기

In [4]:
response = generate_response_using_Llama(
    prompt="Hello world!",
)
print(response)

Hello. What math problem would you like help with today?


#### GSM8K 데이터셋 확인해보기

In [5]:
print("[Question]")
for l in gsm8k_test['question'][0].split("."):
    print(l)
print("="*100)
print("[Answer]")
print(gsm8k_test['answer'][0])

[Question]
Janet’s ducks lay 16 eggs per day
 She eats three for breakfast every morning and bakes muffins for her friends every day with four
 She sells the remainder at the farmers' market daily for $2 per fresh duck egg
 How much in dollars does she make every day at the farmers' market?
[Answer]
Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18


#### Util 함수들
- extract_final_answer: LLM의 응답을 parse하여 최종 결과만 추출 (정답과 비교하기 위해)
- run_benchmark_test: 벤치마크 테스트
- save_final_result: 결과물 제출을 위한 함수

In [6]:
### 수정해도 됩니다!
def extract_final_answer(response: str):
    if "####" in response:
        ans_part = response.split("####")[-1].strip()
        match = re.search(r"(-?\d+(?:\,\d+)?(?:\.\d+)?)", ans_part)
        if match:
            return match.group(1).replace(",", "")

    regex = r"(?:Answer:|The answer is)\s*\$?([0-9,.]+)"
    match = re.search(regex, response, re.IGNORECASE)
    if match:
        return match.group(1).replace(",", "")

    numbers = re.findall(r"(-?\d+(?:\,\d+)?(?:\.\d+)?)", response)
    return numbers[-1].replace(",", "") if numbers else None

In [14]:
### 수정해도 됩니다!
def run_benchmark_test(
        dataset,
        prompt: str,
        model: str = "llama-3.1-8b-instant",
        num_samples: int = 50,
        VERBOSE: bool = False
    ):
    correct = 0
    total   = 0
    results = []

    for i in tqdm(range(min(num_samples, len(dataset)))):
        question = dataset[i]["question"]
        correct_answer = float(re.findall(r'\d+(?:\.\d+)?', dataset[i]["answer"].split('####')[-1])[0])

        time.sleep(1)

        response = generate_response_using_Llama(
            prompt=prompt.format(question=question),
            model=model
        )

        if response:
            if VERBOSE:
                print("="*50)
                print(response)
                print("="*50)
            predicted_answer = extract_final_answer(response)

            if isinstance(predicted_answer, str):
                predicted_answer = float(predicted_answer.replace(",", ""))
            
            diff = abs(predicted_answer - correct_answer)
            is_correct = diff < 1e-5 if predicted_answer is not None else False
            
            if is_correct:
                correct += 1
            total += 1
            
            results.append({
                'question': question,
                'correct_answer': correct_answer,
                'predicted_answer': predicted_answer,
                'response': response,
                'correct': is_correct
            })

            if (i + 1) % 5 == 0:
                current_acc = correct/total if total > 0 else 0
                print(f"Progress: [{i+1}/{num_samples}]")
                print(f"Current Acc.: [{current_acc:.2%}]")

    return results, correct/total if total > 0 else 0

In [8]:
def save_final_result(results: List[Dict[str, Any]], accuracy: float, filename: str) -> None:
    result_str = f"====== ACCURACY: {accuracy} ======\n\n"
    result_str += f"[Details]\n"
    
    for idx, result in enumerate(results):
        result_str += f"Question {idx+1}: {result['question']}\n"
        result_str += f"Correct Answer: {result['correct_answer']}\n"
        result_str += f"Predicted Answer: {result['predicted_answer']}\n"
        result_str += f"Correct: {result['correct']}\n\n"
    
    with open(filename, "w", encoding="utf-8") as f:
        f.write(result_str)

#### Direct prompting with few-shot example

In [42]:
def construct_direct_prompt(num_examples: int = 3) -> str:
    train_dataset = gsm8k_train

    sampled_indices = random.sample(
        [i for i in range(len(train_dataset['question']))],
        num_examples
    )

    prompt = "Instruction:\nSolve the following mathematical question and generate ONLY the answer after a tag, 'Answer:' without any rationale.\n"

    for i in range(num_examples):
        cur_question = train_dataset['question'][i]
        cur_answer = train_dataset['answer'][i].split("####")[-1].strip()

        prompt += f"\n[Example {i+1}]\n"
        prompt += f"Question:\n{cur_question}\n"
        prompt += f"Answer:{cur_answer}\n"

    prompt += "\nQuestion:\n{question}\nAnswer:"

    return prompt

In [43]:
### 어떤 방식으로 저장되는지 확인해보세요!
PROMPT = construct_direct_prompt(3)
VERBOSE = False

results, accuracy = run_benchmark_test(
    dataset=gsm8k_test,
    prompt=PROMPT,
    VERBOSE=VERBOSE,
    num_samples=10
)
save_final_result(results, accuracy, "example.txt")

 50%|█████     | 5/10 [00:07<00:07,  1.60s/it]

Progress: [5/10]
Current Acc.: [60.00%]


100%|██████████| 10/10 [00:15<00:00,  1.52s/it]

Progress: [10/10]
Current Acc.: [60.00%]


In [44]:
# TODO: 0 shot, 3 shot, 5 shot direct prompting을 통해 벤치마크 테스트를 한 후, 각각 direct_prompting_{shot: int}.txt로 저장해주세요!
# 예시: shot이 5인 경우 direct_prompting_5.txt
# 항상 num_samples=50 입니다!
shots = [0, 3, 5]

for shot in shots:
    print(f"\n>>> Running Direct Prompting: {shot}-shot")
    PROMPT = construct_direct_prompt(shot)
    
    results, accuracy = run_benchmark_test(
        dataset=gsm8k_test,
        prompt=PROMPT,
        num_samples=50,
        VERBOSE=False
    )
    
    filename = f"direct_prompting_{shot}.txt"
    save_final_result(results, accuracy, filename)
    print(f"Saved results to {filename} with accuracy: {accuracy}")


>>> Running Direct Prompting: 0-shot


 10%|█         | 5/50 [00:07<01:04,  1.43s/it]

Progress: [5/50]
Current Acc.: [80.00%]


 20%|██        | 10/50 [00:14<01:00,  1.52s/it]

Progress: [10/50]
Current Acc.: [60.00%]


...

Progress: [40/50]
Current Acc.: [75.00%]


 90%|█████████ | 45/50 [01:12<00:07,  1.53s/it]

Progress: [45/50]
Current Acc.: [75.56%]


100%|██████████| 50/50 [01:20<00:00,  1.60s/it]

Progress: [50/50]
Current Acc.: [76.00%]
Saved results to direct_prompting_5.txt with accuracy: 0.76


### Chain-of-Thought prompting with few-shot example
```text
[Question]
Janet’s ducks lay 16 eggs per day
 She eats three for breakfast every morning and bakes muffins for her friends every day with four
 She sells the remainder at the farmers' market daily for $2 per fresh duck egg
 How much in dollars does she make every day at the farmers' market?
====================================================================================================
[Answer]
Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18
```

[Answer] 아래의 정답을 도출하는 과정을 예시로 달아주면 CoT의 few shot이 되겠죠?

In [45]:
def construct_CoT_prompt(num_examples: int = 3) -> str:
    train_dataset = gsm8k_train

    sampled_indices = random.sample(
        [i for i in range(len(train_dataset['question']))],
        num_examples
    )
    
    prompt = (
        "Solve step-by-step. End with #### [value]\n"
    )

    for i, idx in enumerate(sampled_indices):
        cur_question = train_dataset[idx]['question']
        cur_answer = train_dataset[idx]['answer']

        prompt += f"\n[Example {i+1}]\n"
        prompt += f"Question:\n{cur_question}\n"
        prompt += f"Answer:\n{cur_answer}\n"

    prompt += "\nQuestion:\n{question}\nAnswer:"
    
    return prompt

In [46]:
# TODO: 0 shot, 3 shot, 5 shot CoT prompting을 통해 벤치마크 테스트를 한 후, 각각 CoT_prompting_{shot: int}.txt로 저장해주세요!
# 예시: shot이 5인 경우 CoT_prompting_5.txt
# 항상 num_samples=50 입니다!
shots = [0, 3, 5]

for shot in shots:
    print(f"\n>>> Running CoT Prompting: {shot}-shot")
    PROMPT = construct_CoT_prompt(shot)
    
    if shot == 0:
        PROMPT = "Question:\n{question}\nAnswer: Let's think step by step."

    results, accuracy = run_benchmark_test(
        dataset=gsm8k_test,
        prompt=PROMPT,
        num_samples=50,
        VERBOSE=False
    )
    
    filename = f"CoT_prompting_{shot}.txt"
    save_final_result(results, accuracy, filename)
    print(f"Saved results to {filename} with accuracy: {accuracy}")


>>> Running CoT Prompting: 0-shot


 10%|█         | 5/50 [00:08<01:17,  1.72s/it]

Progress: [5/50]
Current Acc.: [60.00%]


 20%|██        | 10/50 [00:17<01:09,  1.74s/it]

Progress: [10/50]
Current Acc.: [60.00%]


...

Progress: [40/50]
Current Acc.: [82.50%]


 90%|█████████ | 45/50 [01:10<00:07,  1.56s/it]

Progress: [45/50]
Current Acc.: [82.22%]


100%|██████████| 50/50 [01:17<00:00,  1.55s/it]

Progress: [50/50]
Current Acc.: [84.00%]
Saved results to CoT_prompting_5.txt with accuracy: 0.84


### Construct your prompt!!

목표: 본인만의 프롬프트를 통해 정답률을 더 끌어올려보기!
- gsm8k의 train 데이터셋에서 예시를 가져온 다음 (자유롭게!)
- 그 예시들에 대한 풀이 과정을 만들어주세요!
- 모든 것들이 자유입니다! Direct Prompting, CoT Prompting을 한 결과보다 정답률만 높으면 돼요.

In [49]:
### 자유롭게 수정해도 됩니다! 완전히 새로 함수를 만들어도 돼요.
def construct_my_prompt(num_examples: int = 3):
    train_dataset = gsm8k_train
    sampled_indices = random.sample(range(len(train_dataset)), num_examples)

    prompt = (
        "Solve the math problem using the DUP method:\n"
        "Stage 1 [Core Question]: Extract the most detailed central goal.\n"
        "Stage 2 [Info]: List all necessary facts related to Stage 1.\n"
        "Stage 3 [Answer]: Solve step-by-step using Stage 1 & 2 info.\n"
        "Final result format: #### [value]\n"
    )

    for i, idx in enumerate(sampled_indices):
        cur_question = train_dataset[idx]['question']
        cur_answer = train_dataset[idx]['answer']
        prompt += f"\nQ:{cur_question}\nA:{cur_answer}\n"
    
    prompt += "\nQ:{question}\nA:"

    return prompt

In [ ]:
# TODO: 만든 0 shot, 3 shot, 5 shot example과 프롬프트를 통해 벤치마크 테스트를 한 후, 각각 My_prompting_{shot: int}.txt로 저장해주세요!
# 예시: shot이 5인 경우 My_prompting_5.txt
# 항상 num_samples=50 입니다!
shots = [0, 3, 5] # 샷 별로 여러 번 돌려보느라 결과 창에는 3 샷만 떠있는데, 0, 3, 5 모두 같은 프롬프트와 같은 코드로 진행하였습니다!
for shot in shots:
    print(f"\n>>> Running My Prompting: {shot}-shot")
    PROMPT = construct_my_prompt(shot)

    if shot == 0:
        PROMPT = PROMPT.format(question="{question}") + " Let's extract the Core Question and Key Info first, then solve."
    
    results, accuracy = run_benchmark_test(
        dataset=gsm8k_test,
        prompt=PROMPT,
        num_samples=50,
        VERBOSE=False
    )
    
    filename = f"My_prompting_{shot}.txt"
    save_final_result(results, accuracy, filename)
    print(f"Saved results to {filename} with accuracy: {accuracy}")


>>> Running My Prompting: 3-shot


 10%|█         | 5/50 [00:08<01:13,  1.64s/it]

Progress: [5/50]
Current Acc.: [100.00%]


 20%|██        | 10/50 [00:23<02:08,  3.21s/it]

Progress: [10/50]
Current Acc.: [80.00%]


...

Progress: [40/50]
Current Acc.: [87.50%]


 90%|█████████ | 45/50 [01:21<00:08,  1.61s/it]

Progress: [45/50]
Current Acc.: [88.89%]


100%|██████████| 50/50 [01:29<00:00,  1.79s/it]

Progress: [50/50]
Current Acc.: [90.00%]
Saved results to My_prompting_3.txt with accuracy: 0.9


### 보고서 작성하기
#### 아래의 내용이 포함되면 됩니다!

1. Direct Prompting, CoT Prompting, My Prompting을 0 shot, 3 shot, 5 shot 정답률을 표로 보여주세요!
2. CoT Prompting이 Direct Prompting에 비해 왜 좋을 수 있는지에 대해서 서술해주세요!
3. 본인이 작성한 프롬프트 기법이 CoT에 비해서 왜 더 좋을 수 있는지에 대해서 설명해주세요!
4. 최종적으로, `PROMPTING.md`에 보고서를 작성해주세요!